In [9]:
%pip install icrawler
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
import glob
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# ==========================================
# 0. Global Configuration and Device Detection (GPU/CPU)
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_DIR = "dataset"

# ImageNet Standard Preprocessing Pipeline
TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"[Init] Current computing device: {DEVICE}")

# ==========================================
# 1. Data Cleaning and Format Conversion
# ==========================================
def clean_existing_dataset():
    """
    Check and clean corrupted files in the dataset/ folder,
    and convert all images to standard RGB Jpeg format to prevent PyTorch reading errors.
    """
    print("\n================== Step 1: Checking and Cleaning Kaggle Image Formats ==================")
    cleaned_count = 0
    valid_count = 0

    for img_path in glob.glob(os.path.join(DATASET_DIR, '*/*')):
        try:
            with Image.open(img_path) as img:
                # Convert to RGB mode (exclude PNG alpha channel RGBA or single-channel grayscale images)
                rgb_img = img.convert('RGB')
                
                # Save uniformly as .jpg
                base_name = os.path.splitext(img_path)[0]
                new_path = base_name + ".jpg"
                rgb_img.save(new_path, "JPEG")
                
                # If the original file was not .jpg, delete the old extension file
                if img_path != new_path and os.path.exists(img_path):
                    os.remove(img_path)
                    
                valid_count += 1
        except Exception as e:
            print(f"Problematic file found, automatically removed: {img_path} (Error: {e})")
            if os.path.exists(img_path):
                os.remove(img_path)
            cleaned_count += 1

    print(f"[Cleaning Complete] Removed {cleaned_count} corrupted files, current valid image count: {valid_count}.\n")

# ==========================================
# 2. Implement 3 Distance Calculation Formulas (Advance Task 2)
# ==========================================
def l1_distance(a, b):
    """Manhattan Distance / L1 Norm"""
    return np.sum(np.abs(a - b), axis=-1)

def l2_distance(a, b):
    """Euclidean Distance / L2 Norm"""
    return np.linalg.norm(a - b, axis=-1)

def cosine_distance(a, b):
    """Cosine Distance = 1 - Cosine Similarity"""
    dot = np.dot(a, b.T)
    norm_a = np.linalg.norm(a, axis=-1, keepdims=True)
    norm_b = np.linalg.norm(b, axis=-1, keepdims=True)
    similarity = dot / (norm_a * norm_b.T + 1e-9)
    return 1.0 - similarity

# ==========================================
# 3. Implement 3 Encoding Model Loaders (Advance Task 1)
# ==========================================
def get_feature_extractor(model_name='resnet18'):
    """Load Pre-trained model and remove the classification head"""
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        extractor = torch.nn.Sequential(*list(model.children())[:-1])
    elif model_name == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier = model.classifier[:-1]
        extractor = model
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier = torch.nn.Identity()
        extractor = model
    else:
        raise ValueError(f"Unsupported model name: {model_name}")
        
    extractor.to(DEVICE)
    extractor.eval()
    return extractor

def extract_single_feature(img_path, extractor):
    """Extract feature vector for a single image"""
    try:
        image = Image.open(img_path).convert('RGB')
        tensor_img = TRANSFORM(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feat = extractor(tensor_img)
            feat = torch.flatten(feat, 1)
        return feat.cpu().numpy().flatten()
    except Exception as e:
        print(f"Feature extraction failed ({img_path}): {e}")
        return None

def extract_batch_features(img_paths, extractor):
    """Extract feature matrix for a batch of images"""
    feats = []
    valid_paths = []
    for p in img_paths:
        f = extract_single_feature(p, extractor)
        if f is not None:
            feats.append(f)
            valid_paths.append(p)
    return np.array(feats), np.array(valid_paths)

# ==========================================
# 4. Basic Task 2: Retrieve Top 5 Similar and Dissimilar Images
# ==========================================
def run_task2_retrieval(query_path, feature_matrix, all_paths, metric='euclidean'):
    print(f"\n[Task 2] Retrieving Top-5 for Query image: {query_path}")
    extractor = get_feature_extractor('resnet18')
    query_feat = extract_single_feature(query_path, extractor)
    
    if metric == 'euclidean':
        dists = l2_distance(feature_matrix, query_feat)
    elif metric == 'manhattan':
        dists = l1_distance(feature_matrix, query_feat)
    elif metric == 'cosine':
        dists = cosine_distance(feature_matrix, query_feat).flatten()
        
    sorted_idx = np.argsort(dists)
    
    # Exclude the query image itself if it exists in the gallery
    if all_paths[sorted_idx[0]] == query_path:
        sorted_idx = sorted_idx[1:]
        
    top5_sim_idx = sorted_idx[:5]
    top5_dissim_idx = sorted_idx[-5:][::-1]
    
    # Plot visualization
    plt.figure(figsize=(15, 7))
    plt.subplot(3, 5, 3)
    plt.imshow(Image.open(query_path))
    plt.title("Query Image", fontweight='bold', color='blue')
    plt.axis('off')
    
    for i, idx in enumerate(top5_sim_idx):
        plt.subplot(3, 5, 6 + i)
        plt.imshow(Image.open(all_paths[idx]))
        plt.title(f"Similar #{i+1}\nDist: {dists[idx]:.2f}", color='green', fontsize=9)
        plt.axis('off')
        
    for i, idx in enumerate(top5_dissim_idx):
        plt.subplot(3, 5, 11 + i)
        plt.imshow(Image.open(all_paths[idx]))
        plt.title(f"Dissimilar #{i+1}\nDist: {dists[idx]:.2f}", color='red', fontsize=9)
        plt.axis('off')
        
    plt.tight_layout()
    plt.savefig('task2_retrieval_result.png')
    plt.close()
    print("[Task 2] Retrieval result image saved as task2_retrieval_result.png")

# ==========================================
# 5. Basic Task 3 & 4: KNN Classifier and Unseen Images Test
# ==========================================
def run_task3_4_knn(X_train, y_train, X_unseen, y_unseen, paths_unseen):
    print("\n[Task 3 & 4] Training KNN classifier and testing on 10 Unseen Images...")
    knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
    knn.fit(X_train, y_train)
    
    y_pred = knn.predict(X_unseen)
    acc = accuracy_score(y_unseen, y_pred)
    print(f"[Task 4] Unseen Images Classification Accuracy: {acc * 100:.2f}%")
    
    class_map_rev = {0: 'Cat', 1: 'Dog'}
    
    plt.figure(figsize=(15, 6))
    for i in range(len(paths_unseen)):
        plt.subplot(2, 5, i + 1)
        plt.imshow(Image.open(paths_unseen[i]))
        
        true_lbl = class_map_rev[y_unseen[i]]
        pred_lbl = class_map_rev[y_pred[i]]
        
        if y_unseen[i] == y_pred[i]:
            t_color = 'green'
            t_str = f"True: {true_lbl}\nPred: {pred_lbl}"
        else:
            t_color = 'red'
            t_str = f"FAILED!\nTrue: {true_lbl}\nPred: {pred_lbl}"
            
        plt.title(t_str, color=t_color, fontweight='bold', fontsize=10)
        plt.axis('off')
        
    plt.suptitle("KNN Prediction on 10 Unseen Images", fontsize=14)
    plt.tight_layout()
    plt.savefig('task4_unseen_predictions.png')
    plt.close()
    print("[Task 4] Prediction result image saved as task4_unseen_predictions.png")

# ==========================================
# 6. Advance Tasks: 3x3 Matrix Cross-Comparison (Advance 1-3)
# ==========================================
def run_advance_experiments(all_img_paths, all_labels):
    print("\n[Advance Tasks] Starting 3 Encoding Models x 3 Distance Metrics comparison using 5-Fold CV...")
    
    models_list = ['resnet18', 'vgg16', 'efficientnet_b0']
    metrics_map = {
        'L1 (Manhattan)': 'manhattan',
        'L2 (Euclidean)': 'euclidean',
        'Cosine': 'cosine'
    }
    
    exp_results = []
    
    # set 5 fold-cross vaildation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for m_name in models_list:
        print(f"\nExtracting features for model [{m_name}]...")
        extractor = get_feature_extractor(m_name)
        
        X_all, valid_paths = extract_batch_features(all_img_paths, extractor)
        y_all = all_labels
        
        if len(X_all) != len(y_all):
            print(f"Warning: Feature extraction mismatch for {m_name}. Skipping...")
            continue

        for metric_disp, metric_code in metrics_map.items():
            fold_accuracies = []
            
            # 5 cross-vaildation
            for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
                X_train, X_test = X_all[train_idx], X_all[test_idx]
                y_train, y_test = y_all[train_idx], y_all[test_idx]
                
                knn = KNeighborsClassifier(n_neighbors=5, metric=metric_code)
                knn.fit(X_train, y_train)
                
                y_pred = knn.predict(X_test)
                acc = accuracy_score(y_test, y_pred)
                fold_accuracies.append(acc)
            
            mean_acc = np.mean(fold_accuracies) * 100
            std_acc = np.std(fold_accuracies) * 100
            
            exp_results.append({
                'Encoding Model': m_name,
                'Feature Dimension': X_all.shape[1],
                'Distance Metric': metric_disp,
                'Mean Accuracy (%)': round(mean_acc, 2),
                'Std Dev (%)': round(std_acc, 2)
            })
            
            print(f"  -> {metric_disp}: Mean Acc = {mean_acc:.2f}%, Std = {std_acc:.2f}%")
            
    df_res = pd.DataFrame(exp_results)
    print("\n================== Experiment Comparison Matrix (5-Fold CV) ==================")
    print(df_res.to_string(index=False))
    
    df_res.to_csv('advance_results.csv', index=False)
    
    plt.figure(figsize=(12, 6))
    
    pivot_df = df_res.pivot(index='Encoding Model', columns='Distance Metric', values='Mean Accuracy (%)')
    pivot_std = df_res.pivot(index='Encoding Model', columns='Distance Metric', values='Std Dev (%)')
    
    ax = pivot_df.plot(kind='bar', yerr=pivot_std, capsize=5, figsize=(12, 6), colormap='viridis')
    
    plt.title('Performance Comparison (3 Models x 3 Metrics) - 5-Fold CV', fontsize=14)
    plt.ylabel('Mean KNN Accuracy (%)')
    plt.ylim(80, 100)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(rotation=0)
    plt.legend(title='Distance Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('advance_comparison_chart.png')
    plt.close()
    print("[Advance] Comparison chart saved as advance_comparison_chart.png, data exported to advance_results.csv")


# ==========================================
# 7. Main Entry Point
# ==========================================
if __name__ == '__main__':
    # 1. Check and clean Kaggle dataset format
    clean_existing_dataset()
    
    # 2. Read dataset paths and labels
    paths_list = []
    labels_list = []
    class_map = {'cat': 0, 'dog': 1}
    
    for cname, label in class_map.items():
        folder = os.path.join(DATASET_DIR, cname)
        for p in glob.glob(os.path.join(folder, '*.*')):
            paths_list.append(p)
            labels_list.append(label)
            
    paths_list = np.array(paths_list)
    labels_list = np.array(labels_list)
    
    if len(paths_list) == 0:
        print("❌ Error: No images found in dataset/ folder! Please ensure the path structure is dataset/cat/ and dataset/dog/.")
    else:
        print(f"✅ Successfully loaded dataset, total {len(paths_list)} images.")
        
        # 3. Build ResNet18 base feature library (for Basic Tasks)
        print("\n[Init] Building ResNet18 base feature library...")
        base_extractor = get_feature_extractor('resnet18')
        base_features, base_paths = extract_batch_features(paths_list, base_extractor)
        
        # 4. Execute Task 2 (Top 5 Similar and Dissimilar Image Retrieval)
        query_image_path = base_paths[0]
        run_task2_retrieval(query_image_path, base_features, base_paths, metric='euclidean')
        
        # 5. Reserve 10 Unseen Images for Task 3 & 4 testing
        X_tr_base, X_unseen, y_tr_base, y_unseen, _, paths_unseen = train_test_split(
            base_features, labels_list, base_paths, test_size=10, stratify=labels_list, random_state=42
        )
        run_task3_4_knn(X_tr_base, y_tr_base, X_unseen, y_unseen, paths_unseen)
        
        # 6. Execute Advance Tasks (Multi-model x Multi-distance Cross-testing)
        run_advance_experiments(paths_list, labels_list)
        
        print("\n🎉 All assignment tasks executed successfully! Result charts and CSV have been generated in the current folder.")

[Init] Current computing device: cpu

================== Step 1: Checking and Cleaning Kaggle Image Formats ==================
[Cleaning Complete] Removed 0 corrupted files, current valid image count: 200.

✅ Successfully loaded dataset, total 200 images.

[Init] Building ResNet18 base feature library...

[Task 2] Retrieving Top-5 for Query image: dataset\cat\cat_168.jpg
[Task 2] Retrieval result image saved as task2_retrieval_result.png

[Task 3 & 4] Training KNN classifier and testing on 10 Unseen Images...
[Task 4] Unseen Images Classification Accuracy: 100.00%
[Task 4] Prediction result image saved as task4_unseen_predictions.png

[Advance Tasks] Starting 3 Encoding Models x 3 Distance Metrics comparison using 5-Fold CV...

Extracting features for model [resnet18]...
  -> L1 (Manhattan): Mean Acc = 97.00%, Std = 2.45%
  -> L2 (Euclidean): Mean Acc = 97.50%, Std = 1.58%
  -> Cosine: Mean Acc = 98.50%, Std = 1.22%

Extracting features for model [vgg16]...
  -> L1 (Manhattan): Mean Ac

<Figure size 1200x600 with 0 Axes>